In [1]:
from rfdetr import RFDETRSmall
import json
import torch
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# --- TRAINING ---

# Initialize model (small variant)
model = RFDETRSmall()

# Optional: capture training metrics
history = []
def on_epoch_end(data):
    history.append(data)

model.callbacks["on_fit_epoch_end"].append(on_epoch_end)

# Train model
model.train(
    dataset_dir="./dataset/brain_tumor",  # directory containing COCO dataset (train/val structure)
    epochs=100,
    batch_size=16,
    lr=1e-4,
)

# Save the model checkpoint
model.save("rf_detr_small_coco.pt")

# Save training history (loss, metrics, etc.)
with open("rf_detr_small_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("✅ Training complete. Model and history saved.")

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights
TensorBoard logging initialized. To monitor logs, use 'tensorboard --logdir output' and open http://localhost:6006/ in browser.
Not using distributed mode
git:
  sha: N/A, status: clean, branch: N/A

Namespace(num_classes=5, grad_accum_steps=4, amp=True, lr=0.0001, lr_encoder=0.00015, batch_size=16, weight_decay=0.0001, epochs=100, lr_drop=100, clip_max_norm=0.1, lr_vit_layer_decay=0.8, lr_component_decay=0.7, do_benchmark=False, dropout=0, drop_path=0.0, drop_mode='standard', drop_schedule='constant', cutoff_epoch=0, pretrained_encoder=None, pretrain_weights='rf-detr-small.pth', pretrain_exclude_keys=None, pretrain_keys_modify_to_l

Epoch: [0]  [  0/114]  eta: 0:56:49  lr: 0.000100  class_error: 62.44  loss: 8.3429 (8.3429)  loss_ce: 1.0784 (1.0784)  loss_bbox: 0.3439 (0.3439)  loss_giou: 0.6310 (0.6310)  loss_ce_0: 1.0802 (1.0802)  loss_bbox_0: 0.3719 (0.3719)  loss_giou_0: 0.6238 (0.6238)  loss_ce_1: 1.0725 (1.0725)  loss_bbox_1: 0.3703 (0.3703)  loss_giou_1: 0.6321 (0.6321)  loss_ce_enc: 1.0389 (1.0389)  loss_bbox_enc: 0.4517 (0.4517)  loss_giou_enc: 0.6482 (0.6482)  loss_ce_unscaled: 1.0784 (1.0784)  class_error_unscaled: 62.4434 (62.4434)  loss_bbox_unscaled: 0.0688 (0.0688)  loss_giou_unscaled: 0.3155 (0.3155)  cardinality_error_unscaled: 3895.5000 (3895.5000)  loss_ce_0_unscaled: 1.0802 (1.0802)  loss_bbox_0_unscaled: 0.0744 (0.0744)  loss_giou_0_unscaled: 0.3119 (0.3119)  cardinality_error_0_unscaled: 3898.3750 (3898.3750)  loss_ce_1_unscaled: 1.0725 (1.0725)  loss_bbox_1_unscaled: 0.0741 (0.0741)  loss_giou_1_unscaled: 0.3161 (0.3161)  cardinality_error_1_unscaled: 3893.5000 (3893.5000)  loss_ce_enc_unsca

In [ ]:
#plot the graphs
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(history)

plt.figure(figsize=(12, 8))

plt.plot(
	df['epoch'],
	df['train_loss'],
	label='Training Loss',
	marker='o',
	linestyle='-'
)

plt.plot(
	df['epoch'],
	df['test_loss'],
	label='Validation Loss',
	marker='o',
	linestyle='--'
)

plt.title('Train/Validation Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# --- INFERENCE / PREDICTION ---
def predict_and_save(model_path, image_path, output_path, conf_threshold=0.5):
    # Load model
    model = RFDETRSmall()
    model.load(model_path)

    # Run prediction
    preds = model.predict(image_path, threshold=conf_threshold)

    # Draw predictions on image
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype("arial.ttf", 14)
    except IOError:
        font = ImageFont.load_default()

    for box, score, label in zip(preds["boxes"], preds["scores"], preds["labels"]):
        x1, y1, x2, y2 = box
        cls_name = preds["class_names"][label]
        caption = f"{cls_name} {score:.2f}"

        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        draw.text((x1, y1 - 12), caption, fill="yellow", font=font)

    image.save(output_path)
    print(f"✅ Saved prediction image with boxes: {output_path}")

# Example usage
predict_and_save(
    model_path="rf_detr_small_coco.pt",
    image_path="sample.jpg",
    output_path="predicted_sample.jpg",
    conf_threshold=0.5
)
